# Phase 2: Hybrid Model Training Pipeline
This notebook contains the production training logic for the hybrid weapon detector, aligning the YOLOv11 backbone, Swin Transformer neck, and Decoupled Focal Head.

### 1. Imports and Environment Setup

In [5]:
import os
import sys
import torch
import torch.optim as optim
from torch.utils.data import DataLoader
from tqdm import tqdm
import numpy as np
from pathlib import Path

# --- Project Root Discovery ---
def get_project_root():
    """Traverses up from the current file to find the project root (identified by requirements.txt)."""
    current_path = Path().resolve()
    for parent in [current_path] + list(current_path.parents):
        if (parent / "requirements.txt").exists():
            return parent
    return current_path

PROJECT_ROOT = get_project_root()
os.chdir(PROJECT_ROOT)  # Sync CWD to root
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

print(f"✅ Environment Aligned. Project Root: {PROJECT_ROOT}")
# ----------------------------------------------

# Project imports 
from models.hybrid_model import HybridWeaponDetector
from ultralytics.data.dataset import YOLODataset
from ultralytics.data.utils import check_det_dataset
from ultralytics.utils import DEFAULT_CFG


✅ Environment Aligned. Project Root: /home/chicken/Desktop/DesktopFiles/DL/AWD&TA/Real-Time-Weapon-Detection-Context-Aware-Red-Alert-System


### 2. Dataset Alignment
We use the real 50k dataset generated in Milestone 1.

In [ ]:
def get_dataloaders(data_yaml_path, batch_size=16, imgsz=640):
    """Initialises real YOLO dataloaders for the hybrid model."""
    # 1. Load dataset config
    data_cfg = check_det_dataset(data_yaml_path)
    
    # 2. Create Train Dataset
    train_set = YOLODataset(
        img_path=data_cfg['train'],
        imgsz=imgsz,
        augment=True,
        batch_size=batch_size,
        task='detect',
        data=data_cfg
    )
    
    # 3. Create Val Dataset
    val_set = YOLODataset(
        img_path=data_cfg['val'],
        imgsz=imgsz,
        augment=False,
        batch_size=batch_size,
        task='detect',
        data=data_cfg
    )
    
    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True, collate_fn=train_set.collate_fn)
    val_loader = DataLoader(val_set, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True, collate_fn=val_set.collate_fn)
    
    return train_loader, val_loader

### 3. Production Training Loop
Implementing the warm-up strategy (frozen backbone) and real loss calculation.

In [ ]:
class HybridTrainer:
    def __init__(self, model, train_loader, val_loader, device="cuda"):
        self.model = model.to(device)
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.device = device
        
        # Loss function (using our custom Focal Head's loss method)
        self.criterion = model.head.compute_loss
        
    def train_epoch(self, optimizer, epoch):
        self.model.train()
        pbar = tqdm(self.train_loader, desc=f"Epoch {epoch}")
        total_loss = 0
        
        for batch in pbar:
            imgs = batch['img'].to(self.device).float() / 255.0
            
            optimizer.zero_grad()
            
            # Forward pass
            preds = self.model(imgs)
            
            # Compute loss (matches ground truth to anchors)
            loss = self.criterion(preds, batch, self.device)
            
            loss.backward()
            optimizer.step()
            
            total_loss += loss.item()
            pbar.set_postfix({"loss": f"{loss.item():.4f}"})
            
        return total_loss / len(self.train_loader)

    def run(self, epochs=50):
        # Phase 1: Freeze Backbone (Warm-up Neck and Head)
        self.model.backbone.freeze()
        optimizer = optim.AdamW(filter(lambda p: p.requires_grad, self.model.parameters()), lr=1e-4)
        
        for epoch in range(1, epochs + 1):
            # Unfreeze backbone halfway through
            if epoch == 10:
                print("\n[INFO] Unfreezing backbone for fine-tuning...")
                self.model.backbone.unfreeze()
                optimizer = optim.AdamW(self.model.parameters(), lr=1e-5)
            
            avg_loss = self.train_epoch(optimizer, epoch)
            print(f"Epoch {epoch} Average Loss: {avg_loss:.4f}")
            
            # Save checkpoint
            if epoch % 5 == 0:
                self.model.save(f"models/weights/epoch_{epoch}.pt")
        
        self.model.save("models/weights/best.pt")

### 4. Execution
Ensure paths are correctly aligned with the project root.

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Executing on: {device}")

# 1. Model Setup
os.makedirs("models/weights", exist_ok=True)
model = HybridWeaponDetector(backbone_variant="yolo11m.pt", pretrained=True, nc=3, device=device)

# 2. Data Setup
data_yaml = r"data/processed/yolo_dataset/data.yaml"
if os.path.exists(data_yaml):
    train_loader, val_loader = get_dataloaders(data_yaml, batch_size=8)
    
    # 3. Training Initiation
    trainer = HybridTrainer(model, train_loader, val_loader, device=device)
    # Start training (example run for 1 epoch to verify logic)
    trainer.run(epochs=1)
    print("\n[SUCCESS] Pipeline aligned. Ready for production training.")
else:
    print(f"[ERROR] data.yaml not found at {data_yaml}. Run setup_data script first.")